# 02 · Unsupervised anomaly detector (Isolation Forest)

Uses **no labels**. It flags payments that look unlike the rest, which is how a pattern
nobody has labelled yet can surface at all. Its decision threshold is set per peer segment in
`03`, because "unusual" depends on who the customer is.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))    # ml/, where sentinel_features.py lives
import sentinel_features as sf

sf.MODELS.mkdir(exist_ok=True)
N = len(sf.FEATURES)
rows = sf.load_rows()        # first run builds the features (~2 min), later runs read the cache
train, test, cutoff = sf.time_split(rows)
print(f"{len(rows):,} rows; train up to {cutoff}, {len(train):,} rows, {train.label.mean():.4%} laundering;"
      f" test {len(test):,} rows, {test.label.mean():.4%} laundering")

In [ ]:
from sklearn.ensemble import IsolationForest

# Trained on a sample of the training period, labels ignored. 1M rows is plenty: each tree
# only ever sees max_samples of them.
fit_rows = sf.matrix(train.sample(1_000_000, random_state=42))
forest = IsolationForest(n_estimators=200, max_samples=512, random_state=42, n_jobs=-1)
forest.fit(fit_rows)

In [ ]:
from sklearn.metrics import average_precision_score

# decision_function: lower = more anomalous. Negated so higher = more suspicious.
scored = test.sample(1_000_000, random_state=7)
anomaly = -forest.decision_function(sf.matrix(scored))
print(f"PR-AUC of the anomaly score against the labels: {average_precision_score(scored.label, anomaly):.3f}"
      f"   (random: {scored.label.mean():.4f})")
print("Expected to be well below XGBoost - it was never shown a label. Its job is the"
      " cases the classifier has no label for.")

In [ ]:
import onnx
import onnxruntime as ort

def attach(model_proto, **metadata):
    """Metadata travels inside the .onnx file, so it cannot drift from the model it describes."""
    for key, value in metadata.items():
        entry = model_proto.metadata_props.add()
        entry.key, entry.value = key, json.dumps(value)
    return model_proto

def save(model_proto, name):
    path = sf.MODELS / f"{name}.onnx"
    onnx.save(model_proto, path)
    (sf.MODELS / f"{name}.features.json").write_text(json.dumps(sf.FEATURES, indent=2) + "\n")
    print(f"wrote {path.name} ({path.stat().st_size / 1024:.0f} KB) and {name}.features.json")
    return ort.InferenceSession(str(path))

# A handful of real rows, fraud included, that scoring-service's parity test replays through
# ONNX Runtime in Java and compares with what Python got here.
parity_rows = pd.concat([test[test.label == 1].head(4), test[test.label == 0].head(4)])
parity_inputs = sf.matrix(parity_rows)

In [ ]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

proto = to_onnx(forest, initial_types=[("features", FloatTensorType([None, N]))],
                target_opset={"": 15, "ai.onnx.ml": 3})
parity_expected = forest.decision_function(parity_inputs)
attach(proto, features=sf.FEATURES,
       parity={"inputs": parity_inputs.tolist(), "score": parity_expected.tolist()})
session = save(proto, "anomaly_isoforest")

sample = sf.matrix(test.sample(100_000, random_state=1))
outputs = dict(zip([o.name for o in session.get_outputs()], session.run(None, {"features": sample})))
print("outputs:", list(outputs))
print("max |onnx - sklearn| =", np.abs(outputs["scores"].ravel() - forest.decision_function(sample)).max())